# NB02: Data Transformation

**LSE ME204 – Data Engineering Principles for the Social Sciences (2026)**

**LSE ID:** 250093214


## Setup

Run the cell below to ensure all required packages are installed before running all other cells.

In [1]:
import json
import requests
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

## What needs to be transformed?

The data does not need a lot of transformation, but there are a couple of things that I would like to do:

- Attach the BSTs of Pokemon onto the species dataframe
    - This was originally done in NB01, but I realized this task would be better suited for NB02
- Filter out certain Pokemon I do not want to include in the analysis
    - This filtering always occurred in this notebook
- Rename the Generations to have actual numbers instead of roman numerals
    - I found, when initially doing my analysis, that having roman numerals caused problems when ordering my graphs. Alphabetically, the roman numeral for 9 (ix) is alphabetically before the roman numeral for 5 (v), so Gen 9 is placed between Gens 4 and 5. This filter was originally applied in NB03, but I felt it prudent to include here
- Cateogorize each Pokemon into one of three categories: Standard, Legendary, Mythical
    - I always knew I wanted to filter by Standard, Legendary, and Mythical Pokemon, but I did not realize I would need a proper categorization attached to each row until I was properly performing my analysis. This categorization originally occurred in NB03, but I thought it better to include here
- Drop columns that are irrelevant in the analysis stage
    - This was never done before my refactoring, but I am doing it now because I think it would make my dataframes look a lot cleaner.

Most of this transformation is going to occur on the species dataframe over the stats dataframe, but transformation will occur on both.

### Loading the data from CSVs

In [2]:
spec_df = pd.read_csv('../data/processed/spec_df.csv')
stats_df = pd.read_csv('../data/processed/stats_df.csv')

### Adding BSTs to the species dataframe

In [3]:
total_bst = (
    stats_df
    .groupby(['name'], as_index = False)
    ['base_stat'].sum()
    .rename(columns={'name':'pokemon_name'})
)

The need for pd.merge(): I have built another dataframe containing the BST for each Pokemon form, automatically sorted alphabetically where the first alphabetical entry is at index 0. This dataframe has different sorting rules than my species dataframe so concatenating the two will not result in the rows being concatenated together. However, they do have a common key: pokemon_name. As such, I can merge them instead and, by using the pokemon_name column as the key to merge on, the right values should go to the right places.

In [ ]:
main_df = pd.merge(left = spec_df, right = total_bst, how = 'left', on = ['pokemon_name'])

,is_default,pokemon_name,pokemon_url,id,name,is_baby,is_legendary,is_mythical,gen,gen_url,base_stat
0,True,bulbasaur,https://pokeapi.co/api/v2/pokemon/1/,1,bulbasaur,False,False,False,generation-i,https://pokeapi.co/api/v2/generation/1/,318
1,True,charmander,https://pokeapi.co/api/v2/pokemon/4/,4,charmander,False,False,False,generation-i,https://pokeapi.co/api/v2/generation/1/,309
2,True,squirtle,https://pokeapi.co/api/v2/pokemon/7/,7,squirtle,False,False,False,generation-i,https://pokeapi.co/api/v2/generation/1/,314
3,True,caterpie,https://pokeapi.co/api/v2/pokemon/10/,10,caterpie,False,False,False,generation-i,https://pokeapi.co/api/v2/generation/1/,195
4,True,weedle,https://pokeapi.co/api/v2/pokemon/13/,13,weedle,False,False,False,generation-i,https://pokeapi.co/api/v2/generation/1/,195
...,...,...,...,...,...,...,...,...,...,...,...
1346,True,arboliva,https://pokeapi.co/api/v2/pokemon/930/,930,arboliva,False,False,False,generation-ix,https://pokeapi.co/api/v2/generation/9/,510
1347,True,revavroom,https://pokeapi.co/api/v2/pokemon/966/,966,revavroom,False,False,False,generation-ix,https://pokeapi.co/api/v2/generation/9/,500
1348,True,arctibax,https://pokeapi.co/api/v2/pokemon/997/,997,arctibax,False,False,False,generation-ix,https://pokeapi.co/api/v2/generation/9/,423
1349,True,baxcalibur,https://pokeapi.co/api/v2/pokemon/998/,998,baxcalibur,False,False,False,generation-ix,https://pokeapi.co/api/v2/generation/9/,600


### Filtering out Pokemon I don't want to include in my analysis

#### What Pokemon will be included? What Pokemon will be excluded?

Pokemon has introduced a lot of gimmicks in its 30 years of existence, and many of these gimmicks affect Pokemon and their BSTs directly: Mega Evolution, Dynamax and Gigantamax, items that make a Pokemon transform into another, more powerful form, and many others. How do we factor in these gimmicks when deciding how what to include?

I think it is perfectly valid to include gimmicks such as Mega Evolution and Gigantimax when discussing power creep, as these are new gimmicks that The Pokemon Company introduced that power up older Pokemon which often get dropped in subsequent Generations. However, I am interested in looking at base Pokemon only, so no Megas or Gigantimaxes will be included. Additionally, I want to compare BST over Generations and see how it has changed, so it does not make sense to include a class of Pokemon that does not grow with each Generation.
Future analyses could consider Mega and Gigantimax forms of Pokemon when looking at power creep from a BST standpoint.

When I say 'base Pokemon,' I mean the default form of that Pokemon. The API has a helpful way of filtering out all non-default form of that Pokemon, and that is through attatching Generation data to only the default form of the Pokemon. All non-default forms (like Megas) have a 'NaN' where the Generation data should be, so I am going to use this as my filter. This filters out a lot of Pokemon that I do not want to include in my analysis, like gimmick Pokemon or Pokemon with multiple forms that have different stat spreads but not different BSTs (ex: Wormadam Plant Cloak, Wormadam Sandy Cloak, Wormadam Trash Cloak). This also excludes Pokemon that I think would be important to include, however, like regional variations of Pokemon (ex: only Kantonian Ratatta is included in the analysis, not Alolan Ratatta), and more powerful versions of Pokemon that were intended to be used by players (ex: the form of Terapagos that is included in the analysis is its Normal Form, which is unuseable in any meaningful way in-game and that has way lower base stats than Stellar Form Terapagos, the form all players will actually use in-game), which has the potential to skew my findings. It may be wise, in a future analysis, to go through and filter the data manually based on stricter criteria that actually gives a better picture of the BSTs of Pokemon that players are using in-game. However, for a first-pass analysis, filtering out all rows with no Generation data attached works well-enough for my purposes. 

Pokemon not included in base generation games but introduced later (ex: in deluxe re-releases, in DLC) will be included in the analysis.

#### How will the filter be applied through code?

Pandas has a built-in .notna() function which goes through and returns a boolean for whether the specified column contains data in a row (True) or not (False). When applied as a filter for a dataset, it will remove rows that contain 'NaN' in the specified column.

If this filter is applied correctly, the dataset should only return 1025 (the number of Pokemon species) rows.

In [5]:
main_df = main_df[main_df['gen'].notna()]

print(len(main_df))

1025


### Transforming roman numerals into arabic numerals

In [6]:
gen_dict = {
    'generation-i':'Gen 1',
    'generation-ii':'Gen 2',
    'generation-iii':'Gen 3',
    'generation-iv':'Gen 4',
    'generation-v':'Gen 5',
    'generation-vi':'Gen 6',
    'generation-vii':'Gen 7',
    'generation-viii':'Gen 8',
    'generation-ix':'Gen 9'
}

In [7]:
def number_gen(generation):
    for key in gen_dict.keys():
        if generation in key:
            return gen_dict[key]

In [8]:
main_df['gen_num'] = main_df['gen'].apply(number_gen)

### Classifying each Pokemon as either Standard, Legendary, or Mythical

This categorization will classify Baby Pokemon as Standard Pokemon, but I am fine with this, as Baby Pokemon are not a class that gets added to every Generation. It does not make sense to examine the change in BST over time for a class of Pokemon that does not grow with every Generation. This was the same logic behind excluding Megas and Gigantimax Pokemon.

In [9]:
def mon_class(row):
    if row['is_legendary'] == False and row['is_mythical'] == False:
        return 'standard'
    elif row['is_legendary'] == True and row['is_mythical'] == False:
        return 'legendary'
    elif row['is_legendary']== False and row['is_mythical'] == True:
        return 'mythical'

In [10]:
main_df['classification'] = main_df.apply(lambda row: mon_class(row), axis = 1)

In [11]:
stats_df

,base_stat,effort,stat_name,stat_url,id,name
0,45,0,hp,https://pokeapi.co/api/v2/stat/1/,1,bulbasaur
1,49,0,attack,https://pokeapi.co/api/v2/stat/2/,1,bulbasaur
2,49,0,defense,https://pokeapi.co/api/v2/stat/3/,1,bulbasaur
3,65,1,special-attack,https://pokeapi.co/api/v2/stat/4/,1,bulbasaur
4,65,0,special-defense,https://pokeapi.co/api/v2/stat/5/,1,bulbasaur
...,...,...,...,...,...,...
8101,175,0,attack,https://pokeapi.co/api/v2/stat/2/,10325,baxcalibur-mega
8102,117,0,defense,https://pokeapi.co/api/v2/stat/3/,10325,baxcalibur-mega
8103,105,0,special-attack,https://pokeapi.co/api/v2/stat/4/,10325,baxcalibur-mega
8104,101,0,special-defense,https://pokeapi.co/api/v2/stat/5/,10325,baxcalibur-mega


### Dropping columns that are unecessary to the analysis

Columns to be dropped in main_df (the species dataframe):
- is_default (All values in this column are True, will not offer any value for analysis)
- pokemon_url, gen_url (I no longer need to call data from the API)
- is_baby, is_legendary, is_mythical (Were good for filtering data to create the classification column, no longer needed with the classification column in place)
- name, gen (contain the same data as other columns, redundant)

Columns to be dropped in stats_df (the stats dataframe):
- effort (Not relevant to the question being answered)
- stat_url (No longer need to call data from the API)

In [12]:
main_df = main_df.drop(columns = ['is_default', 'pokemon_url', 'is_baby', 'is_legendary', 'is_mythical', 'name', 'gen', 'gen_url'])

stats_df = stats_df.drop(columns = ['effort','stat_url'])

I have also decided I want to rename one of the columns to better reflect what data it contains, which is done below.

In [13]:
main_df = main_df.rename(columns = {'base_stat':'BST'})

Now that the data tables have been cleaned up, it is time to store them in csv files for use in NB03

In [14]:
stats_df.to_csv('../data/processed/stats_table.csv', index=False)
main_df.to_csv('../data/processed/base_table.csv', index=False)
